# hello world 最简单的deepagent

In [16]:
from dotenv import load_dotenv
load_dotenv()

True

## 一个演示天气工具调用的 Agent
（返回固定测试文本，不查询真实天气）

In [2]:
import os
from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

# 通过硅基流动接入模型（兼容 OpenAI 接口）
model = ChatOpenAI(
    # 当前免费模型，可用 MODEL_NAME 环境变量覆盖（如付费的 zai-org/GLM-5.2）
    model=os.environ.get("MODEL_NAME", "Qwen/Qwen2.5-7B-Instruct"),
    api_key= os.environ.get("SILICONFLOW_API_KEY"),
    base_url="https://api.siliconflow.cn/v1",
)

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_deep_agent(
    model=model,
    tools=[get_weather],
    system_prompt="You are a helpful assistant.",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "北京今天天气怎么样？"}]}
)

print(result["messages"][-1].content)

今天北京总是晴天！


## 编写自定义工具

In [1]:
from typing import Literal
def internet_search(
    query: str,                          # 1. 参数名 + 类型标注
    max_results: int = 5,                   # 2. 可选参数默认值
    topic: Literal["general", "news", "finance"] = "general",
) -> dict:                               # 返回类型便于理解结果
    """Run a web search for the given query."""  # 3. Docstring
    # 实际的工具逻辑
    return tavily_client.search(query, max_results=max_results, topic=topic)

## 写一个计算器 Agent

In [12]:
def calculate(expression: str) -> float:
    """Evaluate a math expression and return the result.

    Args:
        expression: A math expression, e.g. "1 + 2 * 3".
        Do NOT use variable names or function calls (e.g. use "720.0 * 1.08", NOT "round(720.0 * 1.08, 2)").
    """
    # 只解析数字、括号和四则运算，不执行模型生成的 Python 代码
    import ast
    import operator

    if len(expression) > 200:
        raise ValueError("表达式过长")
    tree = ast.parse(expression, mode="eval")
    if sum(1 for _ in ast.walk(tree)) > 64:
        raise ValueError("表达式过于复杂")
    operations = {ast.Add: operator.add, ast.Sub: operator.sub,
                  ast.Mult: operator.mul, ast.Div: operator.truediv}

    def evaluate(node):
        if isinstance(node, ast.Constant) and type(node.value) in (int, float):
            return float(node.value)
        if isinstance(node, ast.BinOp) and type(node.op) in operations:
            return operations[type(node.op)](evaluate(node.left), evaluate(node.right))
        if isinstance(node, ast.UnaryOp) and isinstance(node.op, (ast.UAdd, ast.USub)):
            value = evaluate(node.operand)
            return value if isinstance(node.op, ast.UAdd) else -value
        raise ValueError(f"不支持的表达式节点 {type(node).__name__},只支持数字、括号和 + - * / 运算")

    return evaluate(tree.body)

def convert_currency(amount: float, from_currency: str, to_currency: str = "CNY") -> dict:
    """Convert an amount from one currency to another.

    Args:
        amount: The amount to convert.
        from_currency: The source currency code, e.g. "USD".
        to_currency: The target currency code, defaults to "CNY".
    """
    # 这里用固定汇率做演示；真实场景可接入汇率 API
    rates = {"USD": 7.2, "CNY": 1.0, "EUR": 7.8}
    cny = amount * rates[from_currency]
    return {"amount": round(cny / rates[to_currency], 2), "currency": to_currency}

In [14]:
agent = create_deep_agent(
    model=model,
    tools=[calculate, convert_currency],
    system_prompt=("你是一个计算助手，能帮用户做数学运算和货币换算。\n"
    "调用 calculate 时，expression 必须是纯数字算式，只能包含数字、小数点、括号和 + - * /。\n"
    "正确示例：\"720.0 * 1.08\"\n"
    "错误示例：\"cny * 1.08\"、\"amount * 1.08\"、\"round(720.0 * 1.08, 2)\"")
)

try:
    result = agent.invoke(
        {"messages": [{"role": "user", "content": "帮我把 100 美元换算成人民币，再用它乘以 1.08 的通胀系数。"}]}
    )
    print(result["messages"][-1].content)
except Exception as e:
    print(f"Error: {e}")

100美元换算成人民币是677元。然后乘以1.08的通胀系数后，结果是731.16元人民币。


# 实战：构建一个研究助手

In [2]:
# 定义搜索工具
import os
from typing import Literal
from tavily import TavilyClient

api_key=os.environ.get("TAVILY_API_KEY")
tavily_client = TavilyClient(api_key=api_key)

def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Run a web search for the given query.

    Args:
        query: The search query string.
        max_results: Maximum number of results to return.
        topic: The topic category for the search.
        include_raw_content: Whether to include raw page content.
    """
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )

In [3]:
# 创建 Agent 并配置系统提示词
from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent
from langchain.agents.middleware import TodoListMiddleware

# 使用硅基流动接入模型（当前免费的 Qwen2.5-7B 即可跑通，复杂任务可改用 GLM-5.2）
model = ChatOpenAI(
    model=os.environ.get("MODEL_NAME", "Qwen/Qwen2.5-7B-Instruct"),  # 可用 MODEL_NAME 覆盖
    api_key=os.environ.get("SILICONFLOW_API_KEY"),
    base_url="https://api.siliconflow.cn/v1",
)

research_instructions = """你是一位专业的研究员。
你的工作是进行深入研究，然后撰写一份完整的研究报告。

你可以使用 internet_search 工具搜索互联网获取信息。
"""

agent = create_deep_agent(
    model=model,
    tools=[internet_search],
    system_prompt=research_instructions,
    middleware=[TodoListMiddleware()],
)

In [4]:
# 运行 Agent
result = agent.invoke(
    {"messages": [{"role": "user", "content": "什么是 LangGraph？"}]}
)

print(result["messages"][-1].content)

LangGraph 是一个由竹间智能开发的大规模可扩展的查询解析和知识抽取平台。它主要用于从大量结构化和非结构化数据中自动提取和理解信息，支持多模态数据处理，并提供自然语言处理、知识图谱构建和查询解析等功能。

LangGraph 平台具有以下特点：

1. 自动化处理能力：能够自动从各种数据源中提取和挖掘有用信息，支持各类文本、音频、视频等多模态数据的处理。
2. 大规模扩展性：能够处理大规模的数据集，适合企业级应用。
3. 智能解析与理解：利用先进的自然语言处理技术，实现对用户的查询语句进行深入理解与解析。
4. 知识图谱构建：能够构建复杂的知识图谱，用于存储和管理各类知识和信息。
5. 查询解析服务：提供高效的查询解析服务，支持复杂的查询语句和参数。

这些能力为开发人员提供了强大的工具来构建更智能的应用程序和服务。

为了获得更详细的信息，我们可以进一步搜索 LangGraph 的相关资料。请问是否需要我为您执行一个网络搜索？
